# 从 HybridML 源码到化工厂灰箱建模：循序渐进的 Notebook

面向化学工程硕士研究生。本教程先读懂仓库中的 `Project`、`ModelCreator`、`NodeRegistry`、模型 JSON 和数据装载器，再分析药代动力学 ODE 示例，最后把同一套“神经网络 + 机理方程 + 按运行验证”的思想映射到放热 CSTR 工厂软测量问题。

**重要：** 仓库附带的 Theophylline 数据是药代动力学示例，不是化工厂数据。Notebook 不会把它说成工业装置验证。这里的工厂案例是工程建模蓝图；若没有工厂历史数据、仪表校验和独立验证，就不能声称模型可直接上线或用于安全控制。

## 学习路径与运行约定

1. 认识源码的职责边界：项目管理、节点注册、JSON 解析/模型构建、数据读取。
2. 从 `simple_model.json` 读懂张量图与命名连线。
3. 从两室 ODE JSON 和 Theophylline CSV 读懂“个体协变量 → ODE 参数 → 时间序列输出”。
4. 将结构迁移到化工厂的非等温 CSTR，并设计不会泄漏的工况验证。

本 Notebook 的代码单元只用 Python 标准库、NumPy、pandas 和 Matplotlib 检查本地文件；不会导入/修改 HybridML 源码，也不需要下载数据。仓库的 HybridML 实现使用旧版 `tensorflow.keras`，ODE 示例还依赖 CasADi；教程环境不一定兼容。需运行原始网络时，应另建符合该仓库依赖版本的环境，而不是在此 Notebook 中静默跳过错误。

In [ ]:
import ast
import json
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 统一图形尺寸，让多条轨迹和文字标签不至于挤在一起。
plt.rcParams['figure.figsize'] = (9, 4)

## 1. 定位仓库并检查教学材料

Notebook 从当前目录向上查找 `HybridML/Project.py`，因此可从仓库根目录或教程目录启动。先明确文件存在，再阅读，不依赖某个固定盘符。

In [ ]:
def find_repo_root(start=Path.cwd()):
    # 当前路径可能是 notebook 所在目录；逐级向上找标志文件，比硬编码磁盘路径可靠。
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'HybridML' / 'Project.py').is_file():
            return candidate
    raise FileNotFoundError(
        '找不到 HybridML/Project.py；请从项目目录或其子目录启动 Jupyter。'
    )


# 把根路径集中保存，后续都从 REPO 拼接，换电脑或目录时无需逐处修改。
REPO = find_repo_root()
FILES = {
    '项目 API': REPO / 'HybridML' / 'Project.py',
    '节点注册表': REPO / 'HybridML' / 'NodeRegistry.py',
    '模型解析/创建': REPO / 'HybridML' / 'ModelCreator.py',
    '数据装载器': REPO / 'HybridML' / 'utils' / 'DataSource.py',
    '节点解析器': REPO / 'HybridML' / 'parsing' / 'nodes' / 'LinearOde.py',
    '节点构建器': REPO / 'HybridML' / 'building' / 'nodes' / 'LinearOde.py',
    'Keras 建图器': REPO / 'HybridML' / 'keras' / 'Builders.py',
    '简单模型 JSON': REPO / 'demo' / 'simple_model.json',
    '两室 ODE JSON': REPO / 'demo' / 'theoph' / 'projects' / 'theoph_demo' / 'theoph_model_2_compartments.json',
    'Theophylline CSV': REPO / 'demo' / 'theoph' / 'projects' / 'theoph_demo' / 'theoph_normalized.csv',
}

# 先做只读的存在性/字节数清点；缺失文件会显示 False，而不是在后续读取时才难定位。
file_inventory = pd.DataFrame([
    {'材料': label, '存在': path.is_file(), '字节': path.stat().st_size if path.is_file() else None}
    for label, path in FILES.items()
])
print('仓库根目录:', REPO)
display(file_inventory)

## 2. 源码各文件分别做什么？

| 文件/组件 | 阅读重点 | 在建模流程中的职责 |
|---|---|---|
| `Project.py` | `Project`、`generate_models`、`load_data_sources` | 管理项目 JSON、模型描述、数据文件与保存目录 |
| `NodeRegistry.py` | parser / builder / custom object 列表 | 告诉系统支持哪些节点类型 |
| `ModelCreator.py` | parser factory、builder factory、`KerasModelCreator` | 将 JSON 依序解析为中间表示，再构建模型 |
| `parsing/nodes/LinearOde.py` | `LinearOdeNodeParser` | 解析 ODE 节点需要的参数和连接 |
| `building/nodes/LinearOde.py` | `LinearOdeNodeBuilder` | 将解析后的节点转换成具体 ODE 层 |
| `keras/Builders.py` | `KerasNetBuilder`、`KerasModelBuilder` | 建 Keras 输入、按依赖次序接线并编译 |
| `utils/DataSource.py` | 文件/时间序列数据装载器 | 按 JSON 指定的列组织输入和目标 |

核心思路是“声明拓扑、注册节点实现、解析依赖、再建图”。源代码以旧版 `tensorflow.keras` 为主；后面的 API 结构导读不执行这些模块，避免把版本不匹配误当成数据问题。

In [ ]:
def list_python_definitions(path):
    # AST 解析语法树而不执行目标源码，因此检查旧版依赖代码时不必导入 TensorFlow/CasADi。
    tree = ast.parse(path.read_text(encoding='utf-8'))
    rows = []
    # tree.body 是模块顶层；类内部的方法再从 class.body 中列出。
    for item in tree.body:
        if isinstance(item, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            methods = [
                child.name for child in getattr(item, 'body', [])
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            rows.append({'定义': item.name, '类型': type(item).__name__, '方法': ', '.join(methods)})
    return pd.DataFrame(rows)


for label in ['项目 API', '节点注册表', '模型解析/创建']:
    print(f'\n{label}: {FILES[label].relative_to(REPO)}')
    display(list_python_definitions(FILES[label]))

### 2.1 阅读关键实现，而不是只看类名

下面从 AST 中提取指定方法的原始代码。观察调用链：`Project.generate_models()` 读取 JSON → `KerasModelCreator` 选择 parser / builder → 每个 node 被解析并接入计算图 → `KerasModelBuilder` 构造最终模型。

In [ ]:
def show_definition(path, name, class_name=None):
    # 返回原始定义文本用于教学阅读，不调用、不改变被检查的仓库源码。
    source = path.read_text(encoding='utf-8')
    tree = ast.parse(source)
    parents = [tree]
    # 若指定 class_name，只在该类的直接成员中找方法，避免同名方法混淆。
    if class_name:
        parents = [
            item for item in tree.body
            if isinstance(item, ast.ClassDef) and item.name == class_name
        ]
    for parent in parents:
        for item in getattr(parent, 'body', []):
            if getattr(item, 'name', None) == name:
                return ast.get_source_segment(source, item)
    raise LookupError(f'未找到 {class_name + "." if class_name else ""}{name} in {path}')


for path, owner, method in [
    (FILES['项目 API'], 'Project', 'generate_models'),
    (FILES['项目 API'], 'Project', 'load_data_sources'),
    (FILES['模型解析/创建'], 'KerasModelCreator', 'generate_model'),
    (FILES['模型解析/创建'], 'KerasModelCreator', '_do_creation'),
]:
    print(f'\n# {owner}.{method}')
    print(show_definition(path, method, owner))

### 2.2 Registry 为什么重要？

`DefaultNodeRegistry` 把节点类型映射到 parser 和 builder。例如 `LinearOdeNodeParser` 解析模型描述，`LinearOdeNodeBuilder` 根据类型选择解析解、CasADi 或 TensorFlow 实现。若 JSON 引入一个新类型但 Registry 没有对应 parser/builder，单纯在 JSON 写一个新名字并不会让框架自动支持它。

In [ ]:
registry_tree = ast.parse(FILES['节点注册表'].read_text(encoding='utf-8'))
registry_class = next(node for node in registry_tree.body if isinstance(node, ast.ClassDef) and node.name == 'DefaultNodeRegistry')
registry_rows = []
for item in registry_class.body:
    if isinstance(item, ast.Assign):
        for target in item.targets:
            if isinstance(target, ast.Name) and target.id in {'node_parsers', 'node_builders', 'custom_objects'}:
                if isinstance(item.value, ast.List):
                    names = [ast.unparse(value) for value in item.value.elts]
                elif isinstance(item.value, ast.Dict):
                    names = [
                        f'{ast.unparse(key)}: {ast.unparse(value)}'
                        for key, value in zip(item.value.keys, item.value.values)
                    ]
                else:
                    names = []
                registry_rows.append({'Registry 项': target.id, '注册内容': ', '.join(names)})
display(pd.DataFrame(registry_rows))

## 3. 从 `simple_model.json` 读懂计算图

HybridML 的 JSON 使用字符串 ID 连接节点：输入张量进入两个小型神经网络，分别生成 `a_out` 与 `b_out`，算术节点再计算 `out = a_out*b_out`。JSON 中的 `additional_outputs` 让训练/诊断时还可观察中间值；模型的训练损失、优化器也由描述指定。这里的 `a`、`b` 是通用占位变量，不应直接解释成某个真实化工量。

In [ ]:
simple = json.loads(FILES['简单模型 JSON'].read_text(encoding='utf-8'))
print('模型:', simple['name'])
print('输入:', [(item['id'], item['size']) for item in simple['inputs']])
print('输出:', simple['outputs'])
print('附加输出:', simple.get('additional_outputs', []))
print('训练配置:', {key: simple.get(key) for key in ['loss', 'optimizer', 'metrics']})

simple_nodes = pd.DataFrame([
    {
        '节点 ID': node['id'],
        '类型': node['type'],
        '输入连接': ', '.join(node.get('inputs', [])),
        '输出连接': ', '.join(node.get('outputs', [])),
        '计算': node.get('expression', f"Dense 层 sizes={[layer['size'] for layer in node.get('layers', [])]}"),
    }
    for node in simple['nodes']
])
display(simple_nodes)

In [ ]:
def draw_model_graph(description):
    nodes = description['nodes']
    positions = {node['id']: (index, 0) for index, node in enumerate(nodes)}
    fig, axis = plt.subplots(figsize=(10, 2.5))
    for node in nodes:
        x, y = positions[node['id']]
        axis.text(x, y, f"{node['id']}\n[{node['type']}]\n→ {', '.join(node.get('outputs', [])) or node.get('expression', '')}",
                  ha='center', va='center', bbox={'boxstyle': 'round,pad=0.5', 'fc': '#e8f1fb'})
        for input_id in node.get('inputs', []):
            source_node = next((item for item in nodes if input_id in item.get('outputs', [])), None)
            if source_node:
                x0 = positions[source_node['id']][0]
                axis.annotate('', xy=(x - 0.1, y), xytext=(x0 + 0.1, y), arrowprops={'arrowstyle': '->'})
            else:
                axis.text(x - 0.35, y - 0.65, f'input: {input_id}', ha='center', fontsize=8)
    axis.set_xlim(-0.7, len(nodes) - 0.3)
    axis.set_ylim(-1.0, 0.8)
    axis.axis('off')
    plt.show()


draw_model_graph(simple)

## 4. 读懂带 ODE 的 Theophylline 示例

这个示例展示了更典型的灰箱结构：体重 `wt` 输入两个正值神经网络，产生吸收/消除参数 `ka`、`ke`；药物剂量构成初始状态；线性 ODE 节点模拟两个隔室，再从状态张量取一个隔室作为输出。系统矩阵为 `[-ka, 0; ka, -ke]`，对应药物从吸收室转入中央室，再从中央室消除的线性动力学近似。

注意：输入序列长度在 JSON 注释中要求一致，但真实样本可能采样时间不完全相同；训练前要确认批次张量的时间网格和填充规则。并且药代动力学不是化工厂过程，本例的价值在于学习“数据估参数、ODE 描述状态演化”的软件结构。

In [ ]:
theoph = json.loads(FILES['两室 ODE JSON'].read_text(encoding='utf-8'))
print('模型:', theoph['name'])
print('输入:', [(item['id'], item['size']) for item in theoph['inputs']])
print('最终输出:', theoph['outputs'], '| 诊断输出:', theoph.get('additional_outputs', []))
for node in theoph['nodes']:
    print(f"\n{node['id']} [{node['type']}]")
    for key in ['inputs', 'outputs', 'layers', 'expression', 'system_matrix', 'initial_value_input', 'time_series_input', 'output']:
        if key in node:
            print(f'  {key}: {node[key]}')

In [ ]:
theoph_df = pd.read_csv(FILES['Theophylline CSV'])
print('数据形状:', theoph_df.shape)
display(theoph_df.head())
print('每个受试者的观测数:')
display(theoph_df.groupby('Subject')['Time'].agg(['count', 'min', 'max']).head())

for subject_id, group in theoph_df.groupby('Subject'):
    plt.plot(group['Time'], group['conc'], alpha=0.45, marker='.', label=f'受试者 {subject_id}')
plt.xlabel('时间（小时）')
plt.ylabel('归一化浓度（文件中的 conc）')
plt.title('仓库 Theophylline 示例数据：个体时间序列')
plt.grid(alpha=0.25)
plt.legend(ncol=3, fontsize=8)
plt.show()

### 4.1 数据装载器与实验设计的关系

`TimeExpandedDataSourceLoader` 按标识列分组：每组的协变量取首行，时间列作为序列输入，目标列作为序列输出。这种接口适合个体/批次轨迹，但应先人工验证每条轨迹的时间网格、重复点、缺失点、量纲和样本间序列长度。

仓库的旧 `FileDataSource` 还包含按行随机打乱和简单比例切分逻辑；对带时间相关性的工厂数据，逐行随机拆分会让同一运行的相邻点同时出现在训练集和测试集，产生信息泄漏。工业泛化评估应以完整批次、生产campaign、设备或时间段为单位留出。

In [ ]:
def split_by_run(frame, run_column='run_id', seed=7, train_fraction=0.6, validation_fraction=0.2):
    # 先按唯一运行 ID 切分，而不是切分行；同一批的时间相邻点不会跑到不同集合。
    run_ids = np.asarray(frame[run_column].drop_duplicates())
    if len(run_ids) < 3:
        raise ValueError('至少需要 3 条独立运行才能划分训练/验证/测试集。')
    # 固定 seed 让同一数据集每次得到相同分组；10 条运行按 60/20/20 约为 6/2/2 条。
    shuffled = np.random.default_rng(seed).permutation(run_ids)
    n_train = max(1, int(len(shuffled) * train_fraction))
    n_validation = max(1, int(len(shuffled) * validation_fraction))
    if n_train + n_validation >= len(shuffled):
        raise ValueError('比例设置必须为测试集保留至少一条完整运行。')
    groups = {
        'train': set(shuffled[:n_train]),
        'validation': set(shuffled[n_train:n_train + n_validation]),
        'test': set(shuffled[n_train + n_validation:]),
    }
    # 集合交集应为空；若非空说明同一 run_id 泄漏到了多个集合。
    if any(groups[a] & groups[b] for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]):
        raise AssertionError('运行 ID 泄漏')
    return groups


# 这里仅用 ID 展示划分逻辑，不是化工厂数据或模型成绩。
run_split_example = split_by_run(pd.DataFrame({'run_id': [f'run-{i}' for i in range(10)]}))
{name: sorted(ids) for name, ids in run_split_example.items()}

## 5. 工程案例蓝图：放热 CSTR 的质量软测量

### 5.1 问题定义

假设工厂希望在实验室化验结果返回前估计连续搅拌釜出口未反应物浓度 `C_A`，同时预测釜温供操作员参考。**这是一份建模蓝图，不代表某个未提供数据的真实工厂已验证模型。** 现场确认之前，先与工艺人员核实反应网络、物性、仪表位置和单位。

恒容、完全混合、单一不可逆放热反应的最低阶动态模型可以写成：

`r = k_ref * exp[-E_a/R * (1/T - 1/T_ref)] * C_A * exp(delta_theta(z))`

`dC_A/dt = (C_A0 - C_A)/tau - r`

`dT/dt = (T_in - T)/tau + (-DeltaH_r)/(rho Cp) * r + UA/(rho Cp V) * (T_c - T)`

`delta_theta(z)` 是受限的神经网络速率修正；乘法指数结构保持速率为正。状态是 `[C_A, T]`；操作量可含 `C_A0`、进料温度 `T_in`、冷却温度 `T_c`、流量/停留时间 `tau`。如有有效的流量、搅拌、催化剂、在线光谱或反应热数据，也可作为明确校验后的特征。

**代入数字：** 假设 `C_A0=1.1 mol/L`、`C_A=0.8 mol/L`、`tau=5 min`、`r=0.03 mol/(L·min)`，则 `dC_A/dt=(1.1-0.8)/5-0.03=0.03 mol/(L·min)`。也就是说入口补给带来 `+0.06`，反应消耗 `-0.03`，净浓度仍以每分钟 `0.03 mol/L` 上升。若速率修正 `delta=0.1`，反应速率乘 `exp(0.1)≈1.105`，不是直接给浓度加 10%。

这些方程是教学习题的简化假设。真实反应器通常还需考虑多反应、活性衰减、传质、可变体积/密度/热容、夹套动态、压力、死区、测量延迟及设备安全限制；需要扩充模型并由独立数据验证。

In [ ]:
# 用结构化表格先约定列名、单位和审查问题；数据入模前每一列都要有明确含义。
plant_schema = pd.DataFrame([
    ('run_id', '批次/campaign/稳定运行的唯一标识', '分组验证，禁止跨集合泄漏'),
    ('time_min', '统一时钟下的采样时间', '对齐实验室样本、DCS 和控制动作'),
    ('c_a0_mol_L', '反应物进料浓度', '定义来源/测量方法和不确定度'),
    ('feed_temperature_K', '进料温度', '绝对温标 K；核验仪表位置'),
    ('coolant_temperature_K', '冷却介质温度', '注明入口/出口或等效温度'),
    ('residence_time_min', '停留时间或流量换算量', '计算式必须和体积/流量基准一致'),
    ('reactor_temperature_K', '釜温状态测量', '核对热套管延迟与校准'),
    ('c_a_out_mol_L', '实验室/在线出口浓度目标', '记录取样时刻、检出限与分析误差'),
], columns=['字段', '工程含义', '建模前审查'])
display(plant_schema)

### 5.2 将工艺问题映射回 HybridML 组件

| 工艺/建模部分 | HybridML 中可对应的机制 | 要进一步实现/确认的事项 |
|---|---|---|
| 进料与操作变量 | JSON `inputs`、数据源 `properties` | 统一量纲、基准、缩放；不能泄漏未来控制动作 |
| 数据驱动的动力学参数 | `nn` 节点 | 网络输出参数/速率修正而非任意替代守恒方程；校验正值与范围 |
| 物料/能量状态演化 | ODE 节点 | CSTR RHS 为非线性；不能直接把两室常系数 `linear_ode` 当作热反应器模型 |
| 状态初值与输出 | `initial_value_input`、ODE `output`、`outputs` | 按运行对齐初值和时间序列；区分训练目标与可观测辅助输出 |
| 训练与可追溯性 | `loss`、`optimizer`、Project JSON | 保存配置、版本、数据窗口、训练/验证运行清单与单位 |

JSON 只描述计算图，不会自动保证热力学正确、量纲一致或安全。仓库已有 `ode`/`linear_ode`/`casadi_linear_ode` 节点也不等于一个完整工业 CSTR 实现。要把 CSTR 接入框架，需先确认节点支持的 RHS 语法、梯度、求解器稳定性、边界/约束处理和依赖版本；必要时实现专用节点并配套单元测试。

## 6. 现场建模与验证的研究生实践流程

1. **定义系统边界与目标：** 明确预测的是出口浓度、转化率、温度还是产品质量；标注测点和响应延迟。
2. **建立可审查的数据字典：** 每列写清单位、采样频率、来源、质量码、缺失/检出限及变换；统一时间戳和运行 ID。
3. **按运行切分：** 训练、调参验证和最终测试按批次/campaign/时间窗隔离；保留异常或切换工况单独评估，不用随机行切分替代外推测试。
4. **先验证机理基线：** 做物料/能量衡算残差、单位检查、参数范围与积分步长收敛检查；检查参数可辨识性和不同参数之间的补偿。
5. **再增加受限数据修正：** 只修正可信机理中的未知项，逐步增加模型复杂度；与机理、纯数据基线同时比较。
6. **报告工程相关误差：** 分别报告浓度/温度误差、不同工况/产品牌号/设备的误差、漂移和置信区间；误差阈值由工艺风险评估确定，不从示例臆造。
7. **部署前独立审查：** 在线影子运行、异常输入/传感器故障测试、回退策略、版本监控和工艺/安全批准；机器学习模型不可替代 SIS/联锁或操作规程。

### 练习

- 在两室 JSON 中标出哪一层产生 `ka`、哪一层编码状态转移；解释 `additional_outputs` 可用于什么诊断。
- 假设把一个连续反应器的时间点随机分到训练集和测试集，指出至少两个可能的信息泄漏来源，并改成按运行隔离。
- 为目标 CSTR 写出各状态/操作量单位和 `dC_A/dt`、`dT/dt` 的量纲核对表；对每个机理假设列出一项可用工厂数据检验的证据。
- 解释为何一条稳态操作轨迹往往不足以同时辨识反应速率、反应热和传热系数；提出能区分这些参数的安全实验设计问题（由工程团队审核）。

## 7. 本仓库源码与示例索引

- 项目入口：[HybridML/Project.py](../../HybridML/Project.py)
- 节点注册：[HybridML/NodeRegistry.py](../../HybridML/NodeRegistry.py)
- JSON 解析与模型创建：[HybridML/ModelCreator.py](../../HybridML/ModelCreator.py)
- 时间序列数据读取：[HybridML/utils/DataSource.py](../../HybridML/utils/DataSource.py)
- 线性 ODE 节点解析/构建：[HybridML/parsing/nodes/LinearOde.py](../../HybridML/parsing/nodes/LinearOde.py)、[HybridML/building/nodes/LinearOde.py](../../HybridML/building/nodes/LinearOde.py)
- Keras 网络构建：[HybridML/keras/Builders.py](../../HybridML/keras/Builders.py)
- 示例模型：[demo/simple_model.json](../../demo/simple_model.json)、[Theophylline ODE JSON](../../demo/theoph/projects/theoph_demo/theoph_model_2_compartments.json)
- 示例数据：[Theophylline CSV](../../demo/theoph/projects/theoph_demo/theoph_normalized.csv)
- 软件背景与原始演示说明：[README.md](../../README.md)

本 Notebook 对源码实现做教学性摘读；如要使用或修改代码，请回到相应源文件、依赖声明和项目许可证核实版本及约束。